# Numerical Methods for Quant Finance Interviews

Numerical methods are the computational backbone of quantitative finance. Every pricing model, risk calculation, and portfolio optimization ultimately relies on numerical algorithms. This notebook covers the core techniques you need to master for quant interviews at firms like Citadel, Two Sigma, Jane Street, DE Shaw, and Goldman Sachs.

**Prerequisites:** Calculus (single and multivariable), linear algebra, basic probability.

**How to use this notebook:** Work through each section carefully. The methods here are interconnected --- root finding appears inside Monte Carlo, linear algebra underpins PDE solvers, and optimization ties everything together. Pay attention to convergence rates and error analysis, as interviewers frequently test these.

---

## 1. Root Finding

Root finding is the problem of solving $f(x) = 0$ for $x$. In finance, this appears constantly: computing implied volatility from option prices (inverting Black-Scholes), finding internal rates of return, calibrating yield curves, and solving for break-even points. The three classical methods differ in their assumptions, convergence speed, and robustness.

### 1.1 Bisection Method

The bisection method is the simplest and most robust root-finding algorithm. It requires only that $f$ is continuous and that we know an interval $[a, b]$ where $f$ changes sign.

**Algorithm:**

1. Start with $[a, b]$ such that $f(a) \cdot f(b) < 0$ (guaranteed root by the Intermediate Value Theorem).
2. Compute the midpoint $c = \frac{a + b}{2}$.
3. If $f(c) = 0$, we found the root. Otherwise:
   - If $f(a) \cdot f(c) < 0$, the root is in $[a, c]$: set $b = c$.
   - If $f(c) \cdot f(b) < 0$, the root is in $[c, b]$: set $a = c$.
4. Repeat until $|b - a| < \epsilon$ for some tolerance $\epsilon$.

**Convergence:** After $n$ iterations, the interval width is $\frac{b - a}{2^n}$. To achieve accuracy $\epsilon$:

$$n \geq \frac{\log(b - a) - \log(\epsilon)}{\log 2}$$

The convergence rate is **linear** --- each iteration adds roughly one bit of accuracy.

**Pros:** Guaranteed to converge; no derivatives needed; extremely robust.  
**Cons:** Slow compared to Newton's method; requires a bracketing interval.

> 💡 **Interview Tip:** Bisection is the standard fallback for computing implied volatility. If asked "how would you find the implied vol?" a safe answer is: "I would use bisection on the Black-Scholes pricing function minus the market price, since the vega is always positive so the function is monotone, guaranteeing a unique root in any reasonable interval."

### 1.2 Newton-Raphson Method

Newton-Raphson uses the derivative to construct a linear approximation and find the root of that approximation, iterating to convergence.

**Algorithm:** Given an initial guess $x_0$, iterate:

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

**Geometric interpretation:** At each step, we draw the tangent line to $f$ at $(x_n, f(x_n))$ and find where it crosses the $x$-axis.

**Convergence analysis:** If $x^*$ is a simple root (i.e., $f'(x^*) \neq 0$) and $f \in C^2$, then near the root:

$$|x_{n+1} - x^*| \leq \frac{|f''(\xi)|}{2|f'(x_n)|} |x_n - x^*|^2$$

for some $\xi$ between $x_n$ and $x^*$. This gives **quadratic convergence** --- the number of correct digits approximately doubles each iteration.

**Derivation via Taylor expansion:** Expand $f$ around $x_n$:

$$0 = f(x^*) \approx f(x_n) + f'(x_n)(x^* - x_n) + \frac{1}{2}f''(x_n)(x^* - x_n)^2$$

Ignoring the quadratic term: $x^* \approx x_n - f(x_n)/f'(x_n)$.

**When Newton fails:**
- $f'(x_n) = 0$ (horizontal tangent --- division by zero)
- Starting too far from the root (may diverge or cycle)
- Multiple roots ($f'(x^*) = 0$): convergence degrades to linear

> 📝 **Example:** Finding implied volatility using Newton-Raphson. Let $C_{\text{BS}}(\sigma)$ be the Black-Scholes call price as a function of volatility $\sigma$. We want to solve $C_{\text{BS}}(\sigma) - C_{\text{market}} = 0$.
>
> The iteration is:
>
> $$\sigma_{n+1} = \sigma_n - \frac{C_{\text{BS}}(\sigma_n) - C_{\text{market}}}{\text{Vega}(\sigma_n)}$$
>
> This converges very quickly (typically 3--5 iterations) because Vega is always positive and the price is smooth and monotone in $\sigma$.

> 💡 **Interview Tip:** Newton-Raphson for implied vol is one of the most commonly asked numerical methods questions. Be prepared to write out the iteration formula and explain why it converges fast (quadratic convergence, smooth monotone function, always positive Vega).

### 1.3 Secant Method

The secant method is a derivative-free variant of Newton-Raphson. Instead of using $f'(x_n)$, it approximates the derivative using two recent function evaluations.

**Algorithm:** Given two initial guesses $x_0$ and $x_1$, iterate:

$$x_{n+1} = x_n - f(x_n) \cdot \frac{x_n - x_{n-1}}{f(x_n) - f(x_{n-1})}$$

**Convergence:** The order of convergence is the **golden ratio** $\varphi = \frac{1 + \sqrt{5}}{2} \approx 1.618$:

$$|x_{n+1} - x^*| \approx C |x_n - x^*|^{1.618}$$

This is **superlinear** but **sub-quadratic** --- slower than Newton but faster than bisection, and requires no derivative computation.

**Comparison of convergence orders:**

| Method | Order | Function evals per step | Derivative needed? |
|:---|:---|:---|:---|
| Bisection | 1 (linear) | 1 | No |
| Secant | $\approx 1.618$ | 1 | No |
| Newton-Raphson | 2 (quadratic) | 1 + 1 derivative | Yes |

> 💡 **Interview Tip:** When comparing methods, discuss the **efficiency index** $p^{1/c}$ where $p$ is the convergence order and $c$ is the cost per step (number of function evaluations). Newton has efficiency $2^{1/2} \approx 1.41$ while the secant method has $1.618^{1/1} = 1.618$ --- the secant method is actually more efficient per function evaluation if derivatives are expensive.

### 1.4 Brent's Method

In practice, production systems often use **Brent's method**, which combines bisection, secant, and inverse quadratic interpolation. It maintains a bracketing interval (guaranteeing convergence like bisection) while achieving superlinear convergence when possible.

Brent's method is the standard choice in most numerical libraries (e.g., `scipy.optimize.brentq`). It is the gold standard for one-dimensional root finding because it combines robustness with speed.

> 💡 **Interview Tip:** If asked what method you would use in production for implied volatility, saying "Brent's method" shows practical awareness. It combines the guaranteed convergence of bisection with the speed of the secant method.

---

## 2. Monte Carlo Methods

Monte Carlo simulation is the workhorse of derivatives pricing and risk management. It is the only practical method for pricing path-dependent, high-dimensional derivatives. The fundamental idea is simple: approximate an expectation by averaging random samples.

**Core formula:** Under the risk-neutral measure $\mathbb{Q}$:

$$V_0 = e^{-rT} \mathbb{E}^\mathbb{Q}[\text{Payoff}(S_T)] \approx e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} \text{Payoff}(S_T^{(i)})$$

where each $S_T^{(i)}$ is a simulated terminal stock price.

### 2.1 Basic Monte Carlo for Option Pricing

For a European call under geometric Brownian motion:

$$S_T = S_0 \exp\!\left[\left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\, Z\right], \quad Z \sim N(0,1)$$

**Algorithm:**
1. Draw $N$ samples $Z_1, Z_2, \ldots, Z_N \sim N(0,1)$.
2. Compute $S_T^{(i)} = S_0 \exp\!\left[(r - \sigma^2/2)T + \sigma\sqrt{T}\, Z_i\right]$ for each $i$.
3. Compute payoffs: $V_i = \max(S_T^{(i)} - K, 0)$.
4. Estimate the price: $\hat{V} = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} V_i$.

**Error analysis:** By the Central Limit Theorem, the standard error of the MC estimate is:

$$\text{SE} = \frac{\sigma_{\text{payoff}}}{\sqrt{N}}$$

where $\sigma_{\text{payoff}}$ is the standard deviation of the discounted payoff. The 95% confidence interval is:

$$\hat{V} \pm 1.96 \cdot \frac{\sigma_{\text{payoff}}}{\sqrt{N}}$$

**Key limitation:** To halve the error, you need $4\times$ as many samples. The convergence rate of $O(1/\sqrt{N})$ is **independent of dimension**, which is why MC is preferred for high-dimensional problems.

> 💡 **Interview Tip:** The $O(1/\sqrt{N})$ convergence rate is the single most important fact about Monte Carlo. Interviewers often ask: "How many more samples do you need to reduce the error by a factor of 10?" Answer: 100 times more, since the error scales as $1/\sqrt{N}$.

### 2.2 Variance Reduction: Antithetic Variates

The idea is to introduce **negative correlation** between samples to reduce variance.

**Method:** For each standard normal draw $Z_i$, also use $-Z_i$. Since $Z$ and $-Z$ both have the same distribution (the normal is symmetric), both give valid samples, but they are negatively correlated.

**Estimator:**

$$\hat{V}_{\text{anti}} = e^{-rT} \cdot \frac{1}{N} \sum_{i=1}^{N} \frac{V(Z_i) + V(-Z_i)}{2}$$

**Variance reduction:** Let $Y_i = \frac{V(Z_i) + V(-Z_i)}{2}$. Then:

$$\text{Var}(Y_i) = \frac{1}{4}\left[\text{Var}(V(Z)) + \text{Var}(V(-Z)) + 2\text{Cov}(V(Z), V(-Z))\right]$$

$$= \frac{1}{2}\text{Var}(V(Z)) + \frac{1}{2}\text{Cov}(V(Z), V(-Z))$$

Since $V(Z)$ and $V(-Z)$ are typically **negatively correlated** (when $Z$ is large, $-Z$ is small), the covariance term is negative, reducing the overall variance.

**When it works best:** When the payoff is a monotone function of $Z$. For a vanilla call, antithetic variates typically reduce variance by 50--75%.

> 💡 **Interview Tip:** Antithetic variates is the simplest variance reduction technique and is almost always used in practice. It is essentially free --- you get twice as many paths with no additional random number generation.

### 2.3 Variance Reduction: Control Variates

The idea is to use a correlated random variable with a **known** expectation to reduce variance.

**Setup:** We want to estimate $\mu = E[V]$ (the option price). Suppose we have a control variate $C$ with known mean $E[C] = \mu_C$. Define the controlled estimator:

$$\hat{V}_{\text{CV}} = \hat{V} - \beta(\hat{C} - \mu_C)$$

where $\hat{V} = \frac{1}{N}\sum V_i$ and $\hat{C} = \frac{1}{N}\sum C_i$.

**Optimal $\beta$:** The variance of the controlled estimator is:

$$\text{Var}(\hat{V}_{\text{CV}}) = \text{Var}(\hat{V})(1 - \rho^2_{V,C})$$

where $\rho_{V,C}$ is the correlation between $V$ and $C$. The optimal coefficient is:

$$\beta^* = \frac{\text{Cov}(V, C)}{\text{Var}(C)}$$

**Variance reduction factor:** $1 - \rho^2$. If $|\rho| = 0.95$, variance is reduced by $\approx 90\%$.

> 📝 **Example:** Pricing an arithmetic Asian call. The arithmetic average has no closed-form price, but the **geometric** average Asian call does (under GBM). Use the geometric Asian price as the control variate. Since arithmetic and geometric averages are highly correlated ($\rho > 0.99$ typically), this achieves enormous variance reduction.

**Common control variates in finance:**
- The **underlying asset** $S_T$ (known expectation $S_0 e^{rT}$ under $\mathbb{Q}$)
- A **vanilla option** on the same underlying (known via Black-Scholes)
- The **geometric average** for arithmetic average options

### 2.4 Variance Reduction: Importance Sampling

Importance sampling changes the probability distribution from which we sample to reduce variance, especially for **rare events** (e.g., deep out-of-the-money options, tail risk).

**Core identity:** For any density $g$ such that $g(x) > 0$ whenever $f(x)h(x) \neq 0$:

$$E_f[h(X)] = \int h(x) f(x)\, dx = \int h(x) \frac{f(x)}{g(x)} g(x)\, dx = E_g\!\left[h(X) \frac{f(X)}{g(X)}\right]$$

The ratio $w(x) = f(x)/g(x)$ is called the **likelihood ratio** or **importance weight**.

**Estimator:**

$$\hat{\mu}_{\text{IS}} = \frac{1}{N} \sum_{i=1}^{N} h(X_i) \cdot \frac{f(X_i)}{g(X_i)}, \quad X_i \sim g$$

**Optimal importance distribution:** The zero-variance distribution is $g^*(x) \propto |h(x)| f(x)$, but this requires knowing the answer (the normalizing constant is $E_f[|h(X)|]$). In practice, we choose $g$ to approximate $g^*$.

> 📝 **Example:** Pricing a deep OTM call with $K \gg S_0$. Under the standard measure, most simulated paths end with $S_T < K$ and contribute zero to the payoff estimate. Instead, shift the drift so that $S_T$ is more likely to exceed $K$:
>
> Sample $Z \sim N(\mu_{\text{shift}}, 1)$ instead of $Z \sim N(0, 1)$, and multiply each payoff by the likelihood ratio:
>
> $$w(Z) = \frac{\phi(Z)}{\phi(Z - \mu_{\text{shift}})} = \exp\!\left(-\mu_{\text{shift}} Z + \frac{\mu_{\text{shift}}^2}{2}\right)$$
>
> This can reduce variance by orders of magnitude for tail events.

> 💡 **Interview Tip:** Importance sampling is critical for risk management (estimating VaR, CVA on deep OTM options). A classic question: "How would you efficiently estimate the probability of a 10-sigma loss?" Answer: use importance sampling with a shifted distribution centered near the event of interest.

### 2.5 Variance Reduction Summary

| Technique | Idea | Variance reduction | When to use |
|:---|:---|:---|:---|
| Antithetic variates | Use $Z$ and $-Z$ | $\sim 50\%$ for monotone payoffs | Always (essentially free) |
| Control variates | Subtract correlated known-mean variable | $1 - \rho^2$ factor | When a correlated closed-form solution exists |
| Importance sampling | Change sampling distribution | Up to orders of magnitude | Rare event estimation, deep OTM options |
| Stratified sampling | Force uniform coverage of sample space | Depends on stratification | When variance is concentrated in subsets |

---

## 3. Finite Difference Methods

Finite differences are used in two key areas: (1) approximating derivatives numerically (for Greeks, gradient computations, etc.) and (2) solving partial differential equations (Black-Scholes PDE, heat equation). Understanding truncation error is essential.

### 3.1 Finite Difference Approximations for Derivatives

Given a smooth function $f(x)$ and step size $h$:

**Forward difference:**

$$f'(x) \approx \frac{f(x+h) - f(x)}{h} + O(h)$$

**Backward difference:**

$$f'(x) \approx \frac{f(x) - f(x-h)}{h} + O(h)$$

**Central difference:**

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} + O(h^2)$$

**Second derivative (central):**

$$f''(x) \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2} + O(h^2)$$

**Derivation via Taylor expansion:** Expand $f(x+h)$ and $f(x-h)$:

$$f(x+h) = f(x) + hf'(x) + \frac{h^2}{2}f''(x) + \frac{h^3}{6}f'''(x) + O(h^4)$$

$$f(x-h) = f(x) - hf'(x) + \frac{h^2}{2}f''(x) - \frac{h^3}{6}f'''(x) + O(h^4)$$

Subtracting: $f(x+h) - f(x-h) = 2hf'(x) + O(h^3)$, so the central difference has error $O(h^2)$.

Adding: $f(x+h) + f(x-h) = 2f(x) + h^2 f''(x) + O(h^4)$, giving the second derivative formula.

> 💡 **Interview Tip:** Central differences are $O(h^2)$ accurate while forward/backward differences are only $O(h)$. However, central differences require **two** function evaluations (at $x+h$ and $x-h$), while forward needs only one new evaluation (if $f(x)$ is already known). In practice, this tradeoff matters for computing Greeks --- use central differences for Delta and Gamma, but forward differences may suffice for quick estimates.

### 3.2 Computing Greeks with Finite Differences

The Greeks (sensitivities of option price to parameters) can be computed numerically when closed-form formulas are unavailable:

**Delta** (sensitivity to underlying price $S$):

$$\Delta = \frac{\partial V}{\partial S} \approx \frac{V(S + \delta S) - V(S - \delta S)}{2\, \delta S}$$

**Gamma** (second-order sensitivity to $S$):

$$\Gamma = \frac{\partial^2 V}{\partial S^2} \approx \frac{V(S + \delta S) - 2V(S) + V(S - \delta S)}{(\delta S)^2}$$

**Vega** (sensitivity to volatility $\sigma$):

$$\mathcal{V} = \frac{\partial V}{\partial \sigma} \approx \frac{V(\sigma + \delta\sigma) - V(\sigma - \delta\sigma)}{2\, \delta\sigma}$$

**Choosing step size $h$:** Too large $\Rightarrow$ truncation error dominates. Too small $\Rightarrow$ floating-point rounding error dominates. The optimal step size balances both:

- For forward/backward differences: $h_{\text{opt}} \sim \sqrt{\epsilon_{\text{machine}}} \approx 10^{-8}$ (for double precision)
- For central differences: $h_{\text{opt}} \sim \epsilon_{\text{machine}}^{1/3} \approx 10^{-5}$

where $\epsilon_{\text{machine}} \approx 2.2 \times 10^{-16}$ for 64-bit floating point.

### 3.3 PDE Solving: The Heat Equation

The Black-Scholes PDE can be transformed into the **heat equation** via a change of variables. Solving the heat equation numerically is therefore directly relevant to option pricing.

**The heat equation:**

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

with initial condition $u(x, 0) = u_0(x)$ and appropriate boundary conditions.

**Discretization:** Let $u_j^n = u(x_j, t_n)$ where $x_j = j \cdot \Delta x$ and $t_n = n \cdot \Delta t$. Apply finite differences:

$$\frac{u_j^{n+1} - u_j^n}{\Delta t} = \alpha \frac{u_{j+1}^n - 2u_j^n + u_{j-1}^n}{(\Delta x)^2}$$

Define $\lambda = \frac{\alpha \, \Delta t}{(\Delta x)^2}$. Then:

$$u_j^{n+1} = \lambda \, u_{j-1}^n + (1 - 2\lambda)\, u_j^n + \lambda \, u_{j+1}^n$$

### 3.4 Explicit vs. Implicit Schemes

**Explicit scheme (Forward Euler):** Uses known values at time $n$ to compute values at time $n+1$.

$$u_j^{n+1} = \lambda \, u_{j-1}^n + (1 - 2\lambda)\, u_j^n + \lambda \, u_{j+1}^n$$

- Simple to implement (no linear system to solve).
- **Stability condition:** $\lambda \leq \frac{1}{2}$, i.e., $\Delta t \leq \frac{(\Delta x)^2}{2\alpha}$. If violated, the solution blows up.

**Implicit scheme (Backward Euler):** Uses unknown values at time $n+1$:

$$u_j^{n+1} - \lambda \, u_{j-1}^{n+1} - \lambda \, u_{j+1}^{n+1} + 2\lambda \, u_j^{n+1} = u_j^n$$

This gives a tridiagonal linear system $A \mathbf{u}^{n+1} = \mathbf{u}^n$ at each time step.

- **Unconditionally stable** (any $\Delta t$ works).
- Requires solving a linear system at each step (but tridiagonal systems are solved in $O(M)$ time via the Thomas algorithm).

**Crank-Nicolson scheme:** Averages explicit and implicit:

$$\frac{u_j^{n+1} - u_j^n}{\Delta t} = \frac{\alpha}{2}\left[\frac{u_{j+1}^n - 2u_j^n + u_{j-1}^n}{(\Delta x)^2} + \frac{u_{j+1}^{n+1} - 2u_j^{n+1} + u_{j-1}^{n+1}}{(\Delta x)^2}\right]$$

- **Unconditionally stable** and **second-order accurate** in both space and time: $O((\Delta x)^2 + (\Delta t)^2)$.
- The standard choice for option pricing PDEs in practice.

| Scheme | Accuracy | Stability | System to solve |
|:---|:---|:---|:---|
| Explicit | $O(\Delta t + (\Delta x)^2)$ | Conditional: $\lambda \leq 1/2$ | None |
| Implicit | $O(\Delta t + (\Delta x)^2)$ | Unconditional | Tridiagonal |
| Crank-Nicolson | $O((\Delta t)^2 + (\Delta x)^2)$ | Unconditional | Tridiagonal |

> 💡 **Interview Tip:** Be prepared to explain why explicit schemes can blow up. The stability condition $\lambda \leq 1/2$ means you need very small time steps for fine spatial grids, making explicit schemes impractical. Crank-Nicolson is the standard in production.

### 3.5 Connection to Black-Scholes

The Black-Scholes PDE is:

$$\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV = 0$$

With the substitution $S = Ke^x$, $\tau = T - t$, and $V = Ke^{-r\tau} v(x, \tau)$, this transforms into the heat equation:

$$\frac{\partial v}{\partial \tau} = \frac{\sigma^2}{2} \frac{\partial^2 v}{\partial x^2} + \left(r - \frac{\sigma^2}{2}\right)\frac{\partial v}{\partial x}$$

A further substitution removes the first-order term, yielding the standard heat equation. This is why finite difference methods for the heat equation directly apply to option pricing.

---

## 4. Numerical Linear Algebra

Linear algebra is the engine room of numerical computing. Portfolio optimization, PDE solvers, regression, and risk calculations all reduce to linear algebra operations. Understanding matrix decompositions and numerical stability is critical.

### 4.1 Cholesky Decomposition

Every **symmetric positive definite** (SPD) matrix $\Sigma$ can be decomposed as:

$$\Sigma = L L^T$$

where $L$ is a **lower triangular** matrix with positive diagonal entries. This is the Cholesky decomposition.

**Algorithm:** The entries of $L$ are computed as:

$$L_{jj} = \sqrt{\Sigma_{jj} - \sum_{k=1}^{j-1} L_{jk}^2}$$

$$L_{ij} = \frac{1}{L_{jj}}\left(\Sigma_{ij} - \sum_{k=1}^{j-1} L_{ik} L_{jk}\right), \quad i > j$$

**Cost:** $O(n^3/3)$ operations --- half the cost of a general LU decomposition.

**Applications in quant finance:**

1. **Generating correlated random variables:** Given a correlation matrix $\Sigma$ and a vector of independent standard normals $\mathbf{Z}$, the vector $\mathbf{X} = L\mathbf{Z}$ has covariance matrix $\Sigma$. This is essential for multi-asset Monte Carlo simulation.

2. **Portfolio optimization:** Solving $\Sigma \mathbf{w} = \mathbf{b}$ via Cholesky is numerically stable and efficient.

3. **Testing positive definiteness:** If Cholesky decomposition fails (negative value under the square root), the matrix is not positive definite. This is a practical diagnostic for covariance matrices.

> 📝 **Example:** Generate two correlated Brownian motions $W_1, W_2$ with correlation $\rho$:
>
> $$\Sigma = \begin{pmatrix} 1 & \rho \\ \rho & 1 \end{pmatrix}, \quad L = \begin{pmatrix} 1 & 0 \\ \rho & \sqrt{1 - \rho^2} \end{pmatrix}$$
>
> Given independent $Z_1, Z_2 \sim N(0,1)$:
>
> $$W_1 = Z_1, \quad W_2 = \rho Z_1 + \sqrt{1 - \rho^2}\, Z_2$$
>
> This is the standard recipe for simulating correlated assets.

> 💡 **Interview Tip:** "How do you generate correlated normals?" is one of the most common quant interview questions. The answer is always Cholesky. Be able to write out the $2 \times 2$ case from memory and explain how it generalizes.

### 4.2 LU Decomposition

Any square matrix $A$ (with appropriate non-singularity conditions) can be decomposed as:

$$PA = LU$$

where $P$ is a permutation matrix, $L$ is lower triangular (with ones on the diagonal), and $U$ is upper triangular.

**Solving $A\mathbf{x} = \mathbf{b}$ via LU:**

1. Decompose: $PA = LU$ (cost: $O(2n^3/3)$).
2. Forward substitution: Solve $L\mathbf{y} = P\mathbf{b}$ (cost: $O(n^2)$).
3. Back substitution: Solve $U\mathbf{x} = \mathbf{y}$ (cost: $O(n^2)$).

**Why LU over Gaussian elimination?** Once the decomposition is computed, solving for multiple right-hand sides $\mathbf{b}$ is cheap ($O(n^2)$ each). This is crucial when solving the same system with different inputs (e.g., pricing multiple options on the same grid).

**Partial pivoting** (choosing the largest pivot element) is essential for numerical stability. Without it, small pivots amplify rounding errors catastrophically.

### 4.3 Condition Numbers

The **condition number** of a matrix measures how sensitive the solution of $A\mathbf{x} = \mathbf{b}$ is to perturbations in $A$ or $\mathbf{b}$.

$$\kappa(A) = \|A\| \cdot \|A^{-1}\|$$

For the 2-norm: $\kappa_2(A) = \frac{\sigma_{\max}}{\sigma_{\min}}$, the ratio of the largest to smallest singular values.

**Interpretation:** If $\kappa(A) = 10^k$, you can expect to lose approximately $k$ digits of accuracy when solving $A\mathbf{x} = \mathbf{b}$.

**Error bound:** For a perturbed system $(A + \delta A)(\mathbf{x} + \delta\mathbf{x}) = \mathbf{b} + \delta\mathbf{b}$:

$$\frac{\|\delta \mathbf{x}\|}{\|\mathbf{x}\|} \leq \kappa(A)\left(\frac{\|\delta A\|}{\|A\|} + \frac{\|\delta \mathbf{b}\|}{\|\mathbf{b}\|}\right)$$

**Properties:**
- $\kappa(A) \geq 1$ for any matrix.
- $\kappa(I) = 1$ (the identity is perfectly conditioned).
- $\kappa(cA) = \kappa(A)$ (scaling does not change the condition number).
- Orthogonal matrices have $\kappa_2(Q) = 1$.

**Ill-conditioned matrices in finance:**
- Covariance matrices of highly correlated assets (near-singular)
- Vandermonde matrices in polynomial fitting
- Large regression design matrices with multicollinearity

> 💡 **Interview Tip:** If asked "how do you know if a numerical result is trustworthy?" a strong answer involves checking the condition number. $\kappa > 10^{10}$ for double precision means the result is essentially meaningless. Remedies include regularization (e.g., Tikhonov/ridge), preconditioning, or using higher precision arithmetic.

### 4.4 Tridiagonal Systems and the Thomas Algorithm

PDE solvers using implicit or Crank-Nicolson schemes produce **tridiagonal** linear systems:

$$\begin{pmatrix} b_1 & c_1 & & \\ a_2 & b_2 & c_2 & \\ & \ddots & \ddots & \ddots \\ & & a_n & b_n \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{pmatrix} = \begin{pmatrix} d_1 \\ d_2 \\ \vdots \\ d_n \end{pmatrix}$$

The **Thomas algorithm** solves this in $O(n)$ time (vs. $O(n^3)$ for general Gaussian elimination):

**Forward sweep:** Eliminate the lower diagonal.

$$c_i' = \frac{c_i}{b_i - a_i c_{i-1}'}, \quad d_i' = \frac{d_i - a_i d_{i-1}'}{b_i - a_i c_{i-1}'}$$

**Back substitution:**

$$x_n = d_n', \quad x_i = d_i' - c_i' x_{i+1}$$

This makes implicit PDE schemes practical for large grids.

---

## 5. Optimization

Optimization is ubiquitous in finance: portfolio optimization (Markowitz), model calibration (fitting parameters to market data), maximum likelihood estimation, and machine learning. Understanding convergence rates and the role of convexity is essential.

### 5.1 Gradient Descent

Gradient descent minimizes a function $f(\mathbf{x})$ by iteratively stepping in the direction of steepest descent:

$$\mathbf{x}_{k+1} = \mathbf{x}_k - \eta_k \nabla f(\mathbf{x}_k)$$

where $\eta_k > 0$ is the **step size** (learning rate).

**Convergence for convex functions:** If $f$ is convex with $L$-Lipschitz continuous gradient ($\|\nabla f(\mathbf{x}) - \nabla f(\mathbf{y})\| \leq L \|\mathbf{x} - \mathbf{y}\|$), then with step size $\eta = 1/L$:

$$f(\mathbf{x}_k) - f(\mathbf{x}^*) \leq \frac{L \|\mathbf{x}_0 - \mathbf{x}^*\|^2}{2k}$$

This is $O(1/k)$ convergence --- **sublinear**.

**Convergence for strongly convex functions:** If $f$ is also $\mu$-strongly convex ($\nabla^2 f \succeq \mu I$), convergence improves to **linear** (geometric):

$$f(\mathbf{x}_k) - f(\mathbf{x}^*) \leq \left(1 - \frac{\mu}{L}\right)^k [f(\mathbf{x}_0) - f(\mathbf{x}^*)]$$

The rate depends on the **condition number** $\kappa = L/\mu$ of the Hessian. Poorly conditioned problems converge slowly.

**Step size selection:**
- **Constant:** $\eta = 1/L$ (safe but conservative).
- **Backtracking line search (Armijo rule):** Start with a large $\eta$ and reduce until sufficient decrease is achieved.
- **Exact line search:** $\eta_k = \arg\min_{\eta} f(\mathbf{x}_k - \eta \nabla f(\mathbf{x}_k))$. Rarely practical but useful for analysis.

> 💡 **Interview Tip:** The convergence rate of gradient descent depends on the condition number $\kappa = L/\mu$ of the objective. If asked "why is gradient descent slow for portfolio optimization?" the answer is that covariance matrices of financial assets are often ill-conditioned (highly correlated assets $\Rightarrow$ large condition number $\Rightarrow$ slow convergence).

### 5.2 Newton's Method for Optimization

Newton's method uses second-order information (the Hessian) to achieve faster convergence. It minimizes the local quadratic approximation at each step.

**Algorithm:** At each iteration, solve the **Newton system**:

$$\mathbf{x}_{k+1} = \mathbf{x}_k - [\nabla^2 f(\mathbf{x}_k)]^{-1} \nabla f(\mathbf{x}_k)$$

**Derivation:** The second-order Taylor approximation of $f$ around $\mathbf{x}_k$ is:

$$f(\mathbf{x}) \approx f(\mathbf{x}_k) + \nabla f(\mathbf{x}_k)^T (\mathbf{x} - \mathbf{x}_k) + \frac{1}{2}(\mathbf{x} - \mathbf{x}_k)^T \nabla^2 f(\mathbf{x}_k) (\mathbf{x} - \mathbf{x}_k)$$

Setting the gradient of this approximation to zero gives the Newton step.

**Convergence:** Near the optimum, Newton's method converges **quadratically**:

$$\|\mathbf{x}_{k+1} - \mathbf{x}^*\| \leq C \|\mathbf{x}_k - \mathbf{x}^*\|^2$$

The number of correct digits doubles each iteration (same as Newton-Raphson for root finding --- in fact, Newton's method for optimization IS Newton-Raphson applied to $\nabla f = 0$).

**Pros:**
- Quadratic convergence near the optimum.
- **Affine invariant:** Performance does not depend on coordinate system (unlike gradient descent).
- The condition number of the Hessian does not affect convergence rate.

**Cons:**
- Requires computing and inverting the Hessian: $O(n^2)$ storage, $O(n^3)$ solve per iteration.
- May not converge if started far from the optimum.
- Hessian must be positive definite; otherwise the step direction may not be a descent direction.

> 📝 **Example:** Calibrating a volatility model (e.g., SABR or Heston) to market option prices. The objective is to minimize the sum of squared pricing errors as a function of model parameters. Newton's method converges quickly when the initial guess is close to the calibrated values, which is typically the case when recalibrating daily.

### 5.3 Quasi-Newton Methods (BFGS)

Quasi-Newton methods approximate the Hessian (or its inverse) using only gradient information, achieving **superlinear convergence** without the $O(n^3)$ cost of computing the true Hessian.

**BFGS update:** Maintain an approximation $B_k \approx \nabla^2 f(\mathbf{x}_k)$. After each step, update $B_k$ using the secant condition:

$$B_{k+1} \mathbf{s}_k = \mathbf{y}_k$$

where $\mathbf{s}_k = \mathbf{x}_{k+1} - \mathbf{x}_k$ and $\mathbf{y}_k = \nabla f(\mathbf{x}_{k+1}) - \nabla f(\mathbf{x}_k)$.

**L-BFGS** is a limited-memory variant that stores only the last $m$ updates (typically $m = 5$--$20$) instead of the full $n \times n$ matrix. It is the standard choice for large-scale optimization.

| Method | Per-iteration cost | Convergence | Hessian needed? |
|:---|:---|:---|:---|
| Gradient descent | $O(n)$ | Linear ($O(1/k)$ for convex) | No |
| Newton | $O(n^3)$ | Quadratic | Yes |
| BFGS | $O(n^2)$ | Superlinear | No |
| L-BFGS | $O(mn)$ | Superlinear | No |

### 5.4 Convexity and Optimality Conditions

**First-order necessary condition:** At a local minimum $\mathbf{x}^*$:

$$\nabla f(\mathbf{x}^*) = \mathbf{0}$$

**Second-order sufficient condition:** $\mathbf{x}^*$ is a strict local minimum if:

$$\nabla f(\mathbf{x}^*) = \mathbf{0} \quad \text{and} \quad \nabla^2 f(\mathbf{x}^*) \succ 0 \;\text{(positive definite)}$$

**Convex functions:** A function $f$ is convex if for all $\mathbf{x}, \mathbf{y}$ and $\lambda \in [0,1]$:

$$f(\lambda \mathbf{x} + (1-\lambda)\mathbf{y}) \leq \lambda f(\mathbf{x}) + (1-\lambda) f(\mathbf{y})$$

Equivalently (for twice-differentiable $f$): $\nabla^2 f(\mathbf{x}) \succeq 0$ for all $\mathbf{x}$.

**Why convexity matters:**
- Every local minimum is a global minimum.
- $\nabla f(\mathbf{x}^*) = \mathbf{0}$ is both necessary AND sufficient.
- Gradient descent and Newton's method are guaranteed to converge to the global optimum.

**Markowitz portfolio optimization** is a convex quadratic program:

$$\min_{\mathbf{w}} \frac{1}{2} \mathbf{w}^T \Sigma \mathbf{w} \quad \text{subject to} \quad \boldsymbol{\mu}^T \mathbf{w} \geq r_{\text{target}}, \quad \mathbf{1}^T \mathbf{w} = 1$$

Since $\Sigma$ is positive semi-definite, the objective is convex, guaranteeing a unique global solution.

> 💡 **Interview Tip:** Understanding the role of convexity is critical. If an interviewer asks about calibration difficulties, a sophisticated answer mentions that many financial models (Heston, local vol) have **non-convex** calibration objectives, meaning gradient-based methods may find local minima. Strategies include multiple random restarts, global optimization methods, or regularization.

---

## 6. Convergence Rates Summary

Understanding convergence rates is essential for choosing the right algorithm and estimating computational cost.

| Method | Convergence rate | Error after $n$ steps |
|:---|:---|:---|
| Bisection | Linear (order 1) | $O(2^{-n})$ |
| Secant method | Superlinear (order $\varphi \approx 1.618$) | $O(\epsilon^{\varphi^n})$ |
| Newton-Raphson | Quadratic (order 2) | $O(\epsilon^{2^n})$ |
| Monte Carlo | $O(1/\sqrt{N})$ | Independent of dimension |
| Explicit FD | $O(\Delta t + (\Delta x)^2)$ | Conditionally stable |
| Crank-Nicolson | $O((\Delta t)^2 + (\Delta x)^2)$ | Unconditionally stable |
| Gradient descent (strongly convex) | Linear: $(1 - \mu/L)^k$ | Depends on $\kappa = L/\mu$ |
| Newton (optimization) | Quadratic | Digits double per iteration |

---

## 7. Interview Problems

The following problems test the core numerical methods concepts covered in this notebook. Work through each one carefully --- these represent the types of questions asked at top quant firms.

### Problem 1: Newton-Raphson Convergence Analysis

> **Problem:** You are using Newton-Raphson to find the implied volatility $\sigma$ by solving $C_{\text{BS}}(\sigma) - C_{\text{market}} = 0$. Starting from $\sigma_0 = 0.3$, the first three iterates are $\sigma_1 = 0.2512$, $\sigma_2 = 0.2500003$, $\sigma_3 = 0.2500000$.
>
> (a) Verify that the convergence appears quadratic.  
> (b) Approximately how many iterations would bisection require to achieve the same accuracy, starting from the interval $[0.01, 1.0]$?  
> (c) Why might Newton-Raphson fail for deep OTM options, and how would you handle it?

**Solution:**

**(a)** For quadratic convergence, the ratio $|e_{n+1}|/|e_n|^2$ should be approximately constant, where $e_n = \sigma_n - \sigma^*$ and $\sigma^* = 0.25$.

$$e_0 = 0.3 - 0.25 = 0.05$$
$$e_1 = 0.2512 - 0.25 = 0.0012$$
$$e_2 = 0.0000003$$

Check the ratio:

$$\frac{|e_1|}{|e_0|^2} = \frac{0.0012}{0.0025} = 0.48$$

$$\frac{|e_2|}{|e_1|^2} = \frac{0.0000003}{0.00000144} \approx 0.21$$

The ratios are bounded constants, confirming **quadratic convergence**. Note how the error goes from $10^{-2}$ to $10^{-3}$ to $10^{-7}$ --- the number of correct digits roughly doubles each iteration.

**(b)** The final accuracy is $|e_3| \approx 10^{-7}$. With bisection starting from $[0.01, 1.0]$ (width $0.99$):

$$n \geq \frac{\log(0.99) - \log(10^{-7})}{\log 2} = \frac{\log(0.99 \times 10^7)}{\log 2} \approx \frac{16.1}{0.693} \approx 23 \text{ iterations}$$

Newton-Raphson needed only 3 iterations versus bisection's 23.

**(c)** For deep OTM options, the Vega $\frac{\partial C}{\partial \sigma}$ becomes extremely small, making the Newton step $\Delta\sigma = -\frac{C - C_{\text{market}}}{\text{Vega}}$ very large and potentially causing the iterate to become negative or overshoot wildly. Solutions:
- Use bisection as a fallback (hybrid method like Brent's)
- Clamp the Newton step: $|\Delta\sigma| \leq \sigma_{\max}$
- Use a different parameterization (e.g., log-volatility)
- Use Jaeckel's rational approximation for the initial guess

### Problem 2: Monte Carlo Variance Reduction

> **Problem:** You are pricing a European call option using Monte Carlo with $N = 100{,}000$ paths. The estimated price is $\hat{V} = 10.25$ with a standard error of $0.15$.
>
> (a) What is the 95% confidence interval for the true price?  
> (b) You implement a control variate using the underlying asset price $S_T$. The correlation between the call payoff and $S_T$ is $\rho = 0.92$. What is the new standard error?  
> (c) Alternatively, how many paths would you need without the control variate to match this accuracy?

**Solution:**

**(a)** The 95% confidence interval is:

$$\hat{V} \pm 1.96 \cdot \text{SE} = 10.25 \pm 1.96 \times 0.15 = 10.25 \pm 0.294$$

$$\text{CI} = [9.956, 10.544]$$

**(b)** With the control variate, the variance reduction factor is $1 - \rho^2$:

$$\text{SE}_{\text{CV}} = \text{SE} \cdot \sqrt{1 - \rho^2} = 0.15 \cdot \sqrt{1 - 0.8464} = 0.15 \cdot \sqrt{0.1536} = 0.15 \times 0.392 \approx 0.0588$$

The standard error drops from $0.15$ to $0.059$ --- a reduction of more than $60\%$.

**(c)** To achieve $\text{SE} = 0.0588$ without the control variate, we need:

$$\frac{\sigma}{\sqrt{N_{\text{new}}}} = 0.0588 = \frac{\sigma}{\sqrt{100{,}000}} \cdot \sqrt{1 - \rho^2}$$

$$N_{\text{new}} = \frac{N}{1 - \rho^2} = \frac{100{,}000}{0.1536} \approx 651{,}000 \text{ paths}$$

The control variate is equivalent to running $6.5\times$ more paths --- a huge computational saving.

### Problem 3: Finite Difference Stability

> **Problem:** You are solving the Black-Scholes PDE for a European put option using a finite difference grid with $M = 200$ spatial points and the explicit scheme. The spatial domain is $S \in [0, 400]$ (stock price), $\sigma = 0.3$, and $T = 1$ year.
>
> (a) What is the maximum allowable time step $\Delta t$ for stability?  
> (b) How many time steps are needed?  
> (c) If you switch to Crank-Nicolson, how does this change?

**Solution:**

**(a)** The spatial step size is $\Delta S = 400 / 200 = 2$. For the Black-Scholes PDE, the effective diffusion coefficient at the grid point $S_j$ is $\alpha_j = \frac{1}{2}\sigma^2 S_j^2$. The stability condition for the explicit scheme is:

$$\lambda = \frac{\alpha_{\max} \cdot \Delta t}{(\Delta S)^2} \leq \frac{1}{2}$$

The worst case is at $S_{\max} = 400$: $\alpha_{\max} = \frac{1}{2}(0.3)^2(400)^2 = 7200$.

$$\Delta t \leq \frac{(\Delta S)^2}{2 \alpha_{\max}} = \frac{4}{2 \times 7200} = \frac{1}{3600} \approx 0.000278 \text{ years}$$

**(b)** The number of time steps needed:

$$N_t \geq \frac{T}{\Delta t} = \frac{1}{1/3600} = 3600 \text{ time steps}$$

This is a very large number of time steps for a relatively coarse spatial grid.

**(c)** Crank-Nicolson is **unconditionally stable**, so there is no restriction on $\Delta t$. We can choose $\Delta t$ based purely on accuracy considerations. With $N_t = 200$ time steps (matching the spatial resolution), $\Delta t = 1/200 = 0.005$, which is $18\times$ fewer steps than the explicit scheme requires. Each step requires solving a tridiagonal system ($O(M) = O(200)$), which is very fast.

Total work: Explicit requires $3600 \times 200 = 720{,}000$ operations. Crank-Nicolson requires $200 \times 200 = 40{,}000$ operations --- an $18\times$ speedup.

### Problem 4: Cholesky and Correlated Simulation

> **Problem:** You need to simulate 3 correlated assets for a basket option. The correlation matrix is:
>
> $$\Sigma = \begin{pmatrix} 1 & 0.6 & 0.3 \\ 0.6 & 1 & 0.5 \\ 0.3 & 0.5 & 1 \end{pmatrix}$$
>
> (a) Compute the Cholesky decomposition $\Sigma = LL^T$.  
> (b) Given independent draws $Z_1 = 0.5$, $Z_2 = -1.2$, $Z_3 = 0.8$, compute the correlated draws.  
> (c) How would you detect if a correlation matrix is not positive definite, and what would you do?

**Solution:**

**(a)** Applying the Cholesky algorithm:

**Row 1:**
$$L_{11} = \sqrt{\Sigma_{11}} = \sqrt{1} = 1$$

**Row 2:**
$$L_{21} = \frac{\Sigma_{21}}{L_{11}} = \frac{0.6}{1} = 0.6$$
$$L_{22} = \sqrt{\Sigma_{22} - L_{21}^2} = \sqrt{1 - 0.36} = \sqrt{0.64} = 0.8$$

**Row 3:**
$$L_{31} = \frac{\Sigma_{31}}{L_{11}} = \frac{0.3}{1} = 0.3$$
$$L_{32} = \frac{\Sigma_{32} - L_{31}L_{21}}{L_{22}} = \frac{0.5 - 0.3 \times 0.6}{0.8} = \frac{0.5 - 0.18}{0.8} = \frac{0.32}{0.8} = 0.4$$
$$L_{33} = \sqrt{\Sigma_{33} - L_{31}^2 - L_{32}^2} = \sqrt{1 - 0.09 - 0.16} = \sqrt{0.75} \approx 0.8660$$

$$L = \begin{pmatrix} 1 & 0 & 0 \\ 0.6 & 0.8 & 0 \\ 0.3 & 0.4 & 0.8660 \end{pmatrix}$$

**(b)** The correlated draws are $\mathbf{X} = L\mathbf{Z}$:

$$X_1 = 1 \cdot 0.5 + 0 \cdot (-1.2) + 0 \cdot 0.8 = 0.5$$

$$X_2 = 0.6 \cdot 0.5 + 0.8 \cdot (-1.2) + 0 \cdot 0.8 = 0.3 - 0.96 = -0.66$$

$$X_3 = 0.3 \cdot 0.5 + 0.4 \cdot (-1.2) + 0.8660 \cdot 0.8 = 0.15 - 0.48 + 0.6928 = 0.3628$$

So $\mathbf{X} = (0.5, -0.66, 0.3628)^T$.

**(c)** Detection: attempt the Cholesky decomposition. If at any step $L_{jj}^2 = \Sigma_{jj} - \sum L_{jk}^2 \leq 0$, the matrix is not positive definite. This is the standard numerical test.

Common causes and fixes:
- **Stale data or missing observations:** Clean the data and recompute.
- **Inconsistent pairwise correlations:** Use the **nearest positive definite** matrix (Higham's algorithm): find the SPD matrix closest to $\Sigma$ in the Frobenius norm.
- **Eigenvalue repair:** Compute the eigendecomposition $\Sigma = Q \Lambda Q^T$, set negative eigenvalues to a small positive value $\epsilon > 0$, and reconstruct: $\Sigma_{\text{fixed}} = Q \max(\Lambda, \epsilon I) Q^T$. Then rescale to restore unit diagonal.

---

## Summary: Key Formulas Quick Reference

| Topic | Formula |
|:---|:---|
| Newton-Raphson | $x_{n+1} = x_n - f(x_n)/f'(x_n)$, quadratic convergence |
| Secant method | $x_{n+1} = x_n - f(x_n)(x_n - x_{n-1})/(f(x_n) - f(x_{n-1}))$, order $\varphi$ |
| Bisection iterations | $n \geq \log_2((b-a)/\epsilon)$ |
| MC standard error | $\text{SE} = \sigma/\sqrt{N}$ |
| Control variate variance | $\text{Var}_{\text{CV}} = \text{Var}(1 - \rho^2)$ |
| Central difference | $f'(x) \approx (f(x+h) - f(x-h))/(2h)$, error $O(h^2)$ |
| Second derivative | $f''(x) \approx (f(x+h) - 2f(x) + f(x-h))/h^2$ |
| Explicit stability | $\lambda = \alpha \Delta t/(\Delta x)^2 \leq 1/2$ |
| Cholesky | $\Sigma = LL^T$, cost $O(n^3/3)$ |
| Condition number | $\kappa(A) = \sigma_{\max}/\sigma_{\min}$ |
| Gradient descent | $\mathbf{x}_{k+1} = \mathbf{x}_k - \eta \nabla f(\mathbf{x}_k)$ |
| Newton optimization | $\mathbf{x}_{k+1} = \mathbf{x}_k - [\nabla^2 f]^{-1} \nabla f$ |

---

## Further Reading

- **Glasserman**, *Monte Carlo Methods in Financial Engineering* --- the definitive reference for MC in finance, covering all variance reduction techniques in depth.
- **Duffy**, *Finite Difference Methods in Financial Engineering* --- comprehensive treatment of FD methods for option pricing PDEs.
- **Nocedal & Wright**, *Numerical Optimization* --- the standard reference for gradient descent, Newton methods, and quasi-Newton methods.
- **Trefethen & Bau**, *Numerical Linear Algebra* --- elegant treatment of matrix decompositions, condition numbers, and numerical stability.
- **Joshi**, *C++ Design Patterns and Derivatives Pricing* --- practical implementations of numerical methods in a finance context.